# CIFAR-10 CNN 검증 및 DNN 비교 실험 노트북

이 노트북은 다음 흐름으로 구성됩니다.

1. CIFAR-10 데이터 로드 및 train / validation / test 분리  
2. 커널 붕괴를 줄이기 위한 보수적 설정 적용  
3. DNN, CNN 각각 학습 및 검증  
4. 정확도(Accuracy), 정밀도(Precision), 재현도(Recall), F1 score 계산  
5. 학습 곡선, 혼동행렬, 클래스별 지표, 예측 예시 등 다양한 시각화 수행  
6. 최종적으로 DNN과 CNN 성능 비교

주의:
- 다중분류에서는 Precision / Recall / F1를 `macro average`와 `weighted average`로 함께 보는 것이 안전합니다.
- 아래 표와 그래프의 대표 비교값은 `macro average`를 기본으로 사용합니다.

## 커널 안정성 대응 포인트

이 노트북은 커널 붕괴가 잦은 환경을 고려해 다음처럼 구성했습니다.

- `batch_size=64`로 과도한 GPU 메모리 사용 억제
- `num_workers=0`으로 DataLoader 관련 충돌 가능성 감소
- 매 실험 후 `gc.collect()`와 `torch.cuda.empty_cache()` 수행
- 한 번에 너무 큰 시각화를 몰아서 만들지 않도록 셀 단위 분리
- epoch 수를 보수적으로 시작하고, 필요하면 점진적으로 증가 가능
- `best model`만 저장하여 불필요한 메모리 사용 감소

필요하면 아래 설정 셀에서 `EPOCHS`, `BATCH_SIZE`를 더 낮춰 시작하시면 됩니다.

In [ ]:

# 1. 기본 라이브러리
import os
import gc
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_recall_fscore_support,
    accuracy_score
)

warnings.filterwarnings("ignore")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. 한글 폰트 설정
import matplotlib
matplotlib.rcParams["axes.unicode_minus"] = False

font_candidates = ["Malgun Gothic", "AppleGothic", "NanumGothic", "DejaVu Sans"]
available_fonts = {f.name for f in matplotlib.font_manager.fontManager.ttflist}

for f in font_candidates:
    if f in available_fonts:
        matplotlib.rcParams["font.family"] = f
        print("사용 폰트:", f)
        break
else:
    print("한글 폰트를 찾지 못했습니다. 기본 폰트를 사용합니다.")

# 기본 스타일은 matplotlib 기본값 유지
plt.style.use("default")


In [ ]:

# 3. 설정값
SEED = 42

DATA_ROOT = "./data"
OUTPUT_DIR = Path("./cnn_dnn_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 64
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

VAL_RATIO = 0.1
EPOCHS_DNN = 15
EPOCHS_CNN = 15
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

EARLY_STOPPING_PATIENCE = 5
USE_AMP = torch.cuda.is_available()

# 커널이 불안정하면 아래 값을 더 낮추세요.
# BATCH_SIZE = 32
# EPOCHS_DNN = 10
# EPOCHS_CNN = 10

CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

In [ ]:

# 4. 재현성 고정
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # 완전 재현성을 위해 benchmark는 끄고 deterministic은 켭니다.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 장치:", device)

In [ ]:

# 5. 전처리 정의
# DNN은 flatten 입력을 사용하지만, 원본 이미지는 먼저 동일하게 정규화합니다.
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

# 시각화용 역정규화
def denormalize_image(img_tensor, mean=CIFAR10_MEAN, std=CIFAR10_STD):
    if img_tensor.ndim == 4:
        img_tensor = img_tensor[0]
    img = img_tensor.detach().cpu().float().clone()
    mean_t = torch.tensor(mean).view(3, 1, 1)
    std_t = torch.tensor(std).view(3, 1, 1)
    img = img * std_t + mean_t
    img = img.clamp(0, 1)
    return img.permute(1, 2, 0).numpy()

In [ ]:

# 6. 데이터셋 로드 및 train / val / test 분리
full_train_dataset = datasets.CIFAR10(
    root=DATA_ROOT,
    train=True,
    download=True,
    transform=train_transform
)

test_dataset = datasets.CIFAR10(
    root=DATA_ROOT,
    train=False,
    download=True,
    transform=test_transform
)

train_size = int(len(full_train_dataset) * (1 - VAL_RATIO))
val_size = len(full_train_dataset) - train_size

generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size], generator=generator)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

print(f"Train: {len(train_dataset):,}")
print(f"Val  : {len(val_dataset):,}")
print(f"Test : {len(test_dataset):,}")

In [ ]:

# 7. 클래스 분포 확인
def get_class_counts(dataset, class_names, base_dataset=None):
    counts = {name: 0 for name in class_names}

    if hasattr(dataset, "targets"):
        targets = dataset.targets
    elif hasattr(dataset, "dataset") and hasattr(dataset.dataset, "targets") and hasattr(dataset, "indices"):
        targets = [dataset.dataset.targets[i] for i in dataset.indices]
    else:
        raise ValueError("지원하지 않는 dataset 형식입니다.")

    for t in targets:
        counts[class_names[int(t)]] += 1
    return counts

train_counts = get_class_counts(train_dataset, CLASS_NAMES)
val_counts = get_class_counts(val_dataset, CLASS_NAMES)
test_counts = get_class_counts(test_dataset, CLASS_NAMES)

dist_df = pd.DataFrame({
    "train": train_counts,
    "val": val_counts,
    "test": test_counts
})
display(dist_df.T)

In [ ]:
# 8. 데이터 분포 시각화
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(CLASS_NAMES))
ax.bar(x - 0.25, [train_counts[c] for c in CLASS_NAMES], width=0.25, label="train")
ax.bar(x,         [val_counts[c] for c in CLASS_NAMES],   width=0.25, label="val")
ax.bar(x + 0.25,  [test_counts[c] for c in CLASS_NAMES],  width=0.25, label="test")

ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=30)
ax.set_ylabel("샘플 수")
ax.set_title("CIFAR-10 클래스 분포")
ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
# 9. 샘플 이미지 시각화
def show_sample_images(loader, class_names, n=12):
    images, labels = next(iter(loader))
    n = min(n, len(images))

    fig = plt.figure(figsize=(15, 6))
    for i in range(n):
        ax = plt.subplot(3, 4, i + 1)
        img = denormalize_image(images[i])
        ax.imshow(img)
        ax.set_title(class_names[labels[i].item()])
        ax.axis("off")
    plt.suptitle("학습 샘플 이미지")
    plt.tight_layout()
    plt.show()

show_sample_images(train_loader, CLASS_NAMES, n=12)


In [ ]:

# 10. 모델 정의
class DNNClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32 * 3, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.network(x)


class CNNClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),      # 32 -> 16
            nn.Dropout(0.25),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),      # 16 -> 8
            nn.Dropout(0.25),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),      # 8 -> 4
            nn.Dropout(0.25),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

print(DNNClassifier())
print()
print(CNNClassifier())

In [ ]:

# 11. 공통 유틸 함수
def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def get_optimizer(model, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY):
    return optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

def compute_epoch_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    return {
        "accuracy": acc,
        "precision_macro": p_macro,
        "recall_macro": r_macro,
        "f1_macro": f1_macro,
        "precision_weighted": p_weighted,
        "recall_weighted": r_weighted,
        "f1_weighted": f1_weighted,
    }

In [ ]:

# 12. 학습 / 검증 / 평가 함수
def run_one_epoch(model, loader, criterion, optimizer=None, scaler=None, training=True):
    if training:
        model.train()
    else:
        model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            if USE_AMP and device.type == "cuda":
                with torch.cuda.amp.autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)

            if training:
                if USE_AMP and device.type == "cuda":
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    metric_dict = compute_epoch_metrics(all_labels, all_preds)
    metric_dict["loss"] = epoch_loss
    return metric_dict, np.array(all_labels), np.array(all_preds)


def train_model(model, train_loader, val_loader, model_name="model", epochs=10):
    criterion = nn.CrossEntropyLoss()
    optimizer = get_optimizer(model)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2
    )
    scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and device.type == "cuda"))

    best_val_f1 = -1
    best_epoch = -1
    patience_counter = 0

    history = []

    save_path = OUTPUT_DIR / f"best_{model_name}.pth"

    for epoch in range(1, epochs + 1):
        train_metrics, _, _ = run_one_epoch(
            model, train_loader, criterion, optimizer=optimizer, scaler=scaler, training=True
        )
        val_metrics, _, _ = run_one_epoch(
            model, val_loader, criterion, optimizer=None, scaler=None, training=False
        )

        scheduler.step(val_metrics["f1_macro"])

        current_lr = optimizer.param_groups[0]["lr"]

        row = {
            "epoch": epoch,
            "lr": current_lr,
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["accuracy"],
            "train_precision_macro": train_metrics["precision_macro"],
            "train_recall_macro": train_metrics["recall_macro"],
            "train_f1_macro": train_metrics["f1_macro"],
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["accuracy"],
            "val_precision_macro": val_metrics["precision_macro"],
            "val_recall_macro": val_metrics["recall_macro"],
            "val_f1_macro": val_metrics["f1_macro"],
            "val_precision_weighted": val_metrics["precision_weighted"],
            "val_recall_weighted": val_metrics["recall_weighted"],
            "val_f1_weighted": val_metrics["f1_weighted"],
        }
        history.append(row)

        print(
            f"[{model_name}] Epoch {epoch:02d}/{epochs} | "
            f"Train Loss {row['train_loss']:.4f} | Train Acc {row['train_acc']:.4f} | "
            f"Val Loss {row['val_loss']:.4f} | Val Acc {row['val_acc']:.4f} | "
            f"Val F1(macro) {row['val_f1_macro']:.4f} | LR {current_lr:.6f}"
        )

        if row["val_f1_macro"] > best_val_f1:
            best_val_f1 = row["val_f1_macro"]
            best_epoch = epoch
            patience_counter = 0
            torch.save(model.state_dict(), save_path)
        else:
            patience_counter += 1

        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping 작동: {epoch} epoch에서 종료")
            break

    history_df = pd.DataFrame(history)

    # best model 로드
    model.load_state_dict(torch.load(save_path, map_location=device))
    print(f"최적 epoch: {best_epoch}, 최적 val_f1_macro: {best_val_f1:.4f}")
    return model, history_df


@torch.no_grad()
def evaluate_model(model, loader, model_name="model"):
    criterion = nn.CrossEntropyLoss()
    metrics, y_true, y_pred = run_one_epoch(
        model, loader, criterion, optimizer=None, scaler=None, training=False
    )

    cm = confusion_matrix(y_true, y_pred)
    cls_report = classification_report(
        y_true, y_pred, target_names=CLASS_NAMES, output_dict=True, zero_division=0
    )
    report_df = pd.DataFrame(cls_report).T

    summary = {
        "model": model_name,
        "accuracy": metrics["accuracy"],
        "precision_macro": metrics["precision_macro"],
        "recall_macro": metrics["recall_macro"],
        "f1_macro": metrics["f1_macro"],
        "precision_weighted": metrics["precision_weighted"],
        "recall_weighted": metrics["recall_weighted"],
        "f1_weighted": metrics["f1_weighted"],
        "loss": metrics["loss"]
    }

    return summary, report_df, cm, y_true, y_pred

In [ ]:
# 13. 시각화 함수들
def plot_learning_curves(history_df, model_name="model"):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    for ax in axes.flat:
        ax.grid(True, alpha=0.3)

    # Loss
    axes[0, 0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train_loss")
    axes[0, 0].plot(history_df["epoch"], history_df["val_loss"], marker="s", label="val_loss")
    axes[0, 0].set_title(f"{model_name} Loss Curve")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].set_ylabel("Loss")
    axes[0, 0].legend()

    # Accuracy
    axes[0, 1].plot(history_df["epoch"], history_df["train_acc"], marker="o", label="train_acc")
    axes[0, 1].plot(history_df["epoch"], history_df["val_acc"], marker="s", label="val_acc")
    axes[0, 1].set_title(f"{model_name} Accuracy Curve")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].set_ylabel("Accuracy")
    axes[0, 1].legend()

    # Macro Precision / Recall / F1
    axes[1, 0].plot(history_df["epoch"], history_df["val_precision_macro"], marker="o", label="val_precision_macro")
    axes[1, 0].plot(history_df["epoch"], history_df["val_recall_macro"], marker="s", label="val_recall_macro")
    axes[1, 0].plot(history_df["epoch"], history_df["val_f1_macro"], marker="^", label="val_f1_macro")
    axes[1, 0].set_title(f"{model_name} Validation Macro Metrics")
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].set_ylabel("Score")
    axes[1, 0].legend()

    # LR
    axes[1, 1].plot(history_df["epoch"], history_df["lr"], marker="o")
    axes[1, 1].set_title(f"{model_name} Learning Rate Schedule")
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].set_ylabel("Learning Rate")

    plt.tight_layout()
    plt.show()


def plot_confusion_matrix(cm, class_names, title="Confusion Matrix", cmap="Blues"):
    fig, ax = plt.subplots(figsize=(10, 8))

    im = ax.imshow(cm, interpolation="nearest", cmap=cmap)
    ax.set_title(title)

    tick_marks = np.arange(len(class_names))
    ax.set_xticks(tick_marks)
    ax.set_yticks(tick_marks)
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)

    threshold = cm.max() / 2.0 if cm.size else 0

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            text_color = "white" if cm[i, j] > threshold else "black"
            ax.text(
                j, i, format(cm[i, j], "d"),
                ha="center", va="center",
                color=text_color,
                fontsize=9
            )

    ax.set_ylabel("True label")
    ax.set_xlabel("Predicted label")

    cbar = plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()


def plot_classwise_metrics(report_df, model_name="model"):
    class_df = report_df.loc[CLASS_NAMES, ["precision", "recall", "f1-score"]]

    x = np.arange(len(class_df))
    width = 0.25

    fig, ax = plt.subplots(figsize=(14, 6))

    ax.bar(x - width, class_df["precision"], width=width, label="precision")
    ax.bar(x,         class_df["recall"],    width=width, label="recall")
    ax.bar(x + width, class_df["f1-score"],  width=width, label="f1-score")

    ax.set_xticks(x)
    ax.set_xticklabels(CLASS_NAMES, rotation=30)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title(f"{model_name} 클래스별 Precision / Recall / F1")
    ax.legend()

    plt.tight_layout()
    plt.show()


@torch.no_grad()
def show_prediction_examples(model, loader, class_names, n=12, only_wrong=False, title="Prediction Examples"):
    model.eval()
    shown = 0

    fig = plt.figure(figsize=(15, 8))
    for images, labels in loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1)

        for i in range(len(images)):
            is_wrong = preds[i].item() != labels[i].item()
            if (only_wrong and is_wrong) or (not only_wrong):
                shown += 1
                ax = plt.subplot(math.ceil(n / 4), 4, shown)
                img = denormalize_image(images[i].cpu())
                ax.imshow(img)
                ax.set_title(
                    f"T: {class_names[labels[i].item()]}
P: {class_names[preds[i].item()]}",
                    fontsize=10,
                )
                ax.axis("off")

                if shown >= n:
                    plt.suptitle(title)
                    plt.tight_layout()
                    plt.show()
                    return

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


In [ ]:

# 14. DNN 학습
cleanup_memory()
dnn_model = DNNClassifier(num_classes=10).to(device)
dnn_model, dnn_history = train_model(
    dnn_model, train_loader, val_loader,
    model_name="dnn", epochs=EPOCHS_DNN
)

dnn_history.to_csv(OUTPUT_DIR / "dnn_history.csv", index=False)
display(dnn_history.tail())

In [ ]:

# 15. DNN 학습 곡선 시각화
plot_learning_curves(dnn_history, model_name="DNN")

In [ ]:

# 16. DNN 테스트 평가
dnn_summary, dnn_report_df, dnn_cm, dnn_y_true, dnn_y_pred = evaluate_model(
    dnn_model, test_loader, model_name="DNN"
)

dnn_summary_df = pd.DataFrame([dnn_summary])
display(dnn_summary_df)

display(dnn_report_df.loc[CLASS_NAMES + ["accuracy", "macro avg", "weighted avg"]])

In [ ]:

# 17. DNN 결과 시각화
plot_confusion_matrix(dnn_cm, CLASS_NAMES, title="DNN Confusion Matrix")
plot_classwise_metrics(dnn_report_df, model_name="DNN")
show_prediction_examples(dnn_model, test_loader, CLASS_NAMES, n=12, only_wrong=False, title="DNN 예측 예시")
show_prediction_examples(dnn_model, test_loader, CLASS_NAMES, n=12, only_wrong=True, title="DNN 오답 예시")

In [ ]:

# 18. DNN 메모리 정리 후 CNN 학습
cleanup_memory()
cnn_model = CNNClassifier(num_classes=10).to(device)
cnn_model, cnn_history = train_model(
    cnn_model, train_loader, val_loader,
    model_name="cnn", epochs=EPOCHS_CNN
)

cnn_history.to_csv(OUTPUT_DIR / "cnn_history.csv", index=False)
display(cnn_history.tail())

In [ ]:

# 19. CNN 학습 곡선 시각화
plot_learning_curves(cnn_history, model_name="CNN")

In [ ]:

# 20. CNN 테스트 평가
cnn_summary, cnn_report_df, cnn_cm, cnn_y_true, cnn_y_pred = evaluate_model(
    cnn_model, test_loader, model_name="CNN"
)

cnn_summary_df = pd.DataFrame([cnn_summary])
display(cnn_summary_df)

display(cnn_report_df.loc[CLASS_NAMES + ["accuracy", "macro avg", "weighted avg"]])

In [ ]:

# 21. CNN 결과 시각화
plot_confusion_matrix(cnn_cm, CLASS_NAMES, title="CNN Confusion Matrix")
plot_classwise_metrics(cnn_report_df, model_name="CNN")
show_prediction_examples(cnn_model, test_loader, CLASS_NAMES, n=12, only_wrong=False, title="CNN 예측 예시")
show_prediction_examples(cnn_model, test_loader, CLASS_NAMES, n=12, only_wrong=True, title="CNN 오답 예시")

In [ ]:

# 22. DNN vs CNN 최종 비교 표
comparison_df = pd.DataFrame([dnn_summary, cnn_summary]).set_index("model")
display(comparison_df)

comparison_df.to_csv(OUTPUT_DIR / "dnn_cnn_comparison.csv")

In [ ]:
# 23. DNN vs CNN 핵심 지표 비교 시각화
metrics_to_plot = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(metrics_to_plot))
width = 0.35

dnn_values = comparison_df.loc["DNN", metrics_to_plot].values
cnn_values = comparison_df.loc["CNN", metrics_to_plot].values

ax.bar(x - width/2, dnn_values, width=width, label="DNN")
ax.bar(x + width/2, cnn_values, width=width, label="CNN")

ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot, rotation=20)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("DNN vs CNN 핵심 성능 비교")
ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
# 24. 클래스별 F1 score 비교
dnn_class_f1 = dnn_report_df.loc[CLASS_NAMES, "f1-score"].values
cnn_class_f1 = cnn_report_df.loc[CLASS_NAMES, "f1-score"].values

x = np.arange(len(CLASS_NAMES))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))

ax.bar(x - width/2, dnn_class_f1, width=width, label="DNN")
ax.bar(x + width/2, cnn_class_f1, width=width, label="CNN")
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=30)
ax.set_ylim(0, 1.05)
ax.set_ylabel("F1 score")
ax.set_title("클래스별 F1 score 비교")
ax.legend()

plt.tight_layout()
plt.show()


In [ ]:

# 25. 최종 해석용 요약 테이블
final_table = comparison_df[[
    "accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "precision_weighted",
    "recall_weighted",
    "f1_weighted",
    "loss"
]].copy()

final_table = final_table.sort_values("f1_macro", ascending=False)
display(final_table.style.format("{:.4f}"))

## 결과 해석 가이드

일반적으로 CIFAR-10에서는 CNN이 DNN보다 유리합니다.

이유:
1. CNN은 이미지의 공간 구조를 보존합니다.  
2. 필터(Convolution filter)가 가장자리, 질감, 형태 같은 패턴을 자동으로 학습합니다.  
3. DNN은 이미지를 일렬로 펼쳐서 보기 때문에, 픽셀의 위치 관계 정보가 많이 약해집니다.

따라서 보통 다음 패턴이 나타납니다.

- CNN의 Accuracy, Recall, F1 score가 DNN보다 높음
- DNN은 비슷한 모양의 클래스(cat / dog, automobile / truck 등)에서 더 자주 헷갈림
- CNN은 클래스별 성능 편차가 상대적으로 줄어드는 경우가 많음

## 커널이 다시 불안정할 때 조정할 항목

아래 순서대로 낮추면 됩니다.

1. `BATCH_SIZE = 32`
2. `EPOCHS_DNN = 8`, `EPOCHS_CNN = 8`
3. 오답 예시 시각화 셀은 나중에 실행
4. DNN 학습 완료 후 런타임 재시작 없이 바로 CNN만 돌리지 말고 `cleanup_memory()` 확인
5. 그래도 불안정하면 CNN만 먼저 실행하고, DNN은 이전 결과 CSV와 비교

필요하면 이 노트북을 기준으로 다음 버전도 확장할 수 있습니다.

- 데이터 증강 포함 CNN 버전
- 더 깊은 CNN 버전
- ResNet transfer learning 비교 버전
- 발표용 결과 요약 전용 노트북 버전